# 800 — External Bulk Cohorts

## PREFLIGHT-A — CPTAC-3 cohort eligibility and frozen-input readiness

**Status:** implementation draft; no biological replication analysis has been authorized or executed.

**Objective.** Convert the Notebook 111 CPTAC-3 acquisition audit into auditable, *technical* candidate inventories for prospective external-bulk validation of the three frozen Phase 4 consensus transcriptomic representations.

This initial Notebook 800 implementation is restricted to: (1) referencing the frozen acquisition evidence; (2) auditing primary-tumor candidate coverage and case multiplicity; (3) checking *lexical* overlap against the frozen TCGA sample mapping; (4) inspecting a deterministic, technical STAR Counts gene-ID pilot; and (5) writing non-frozen preflight diagnostics. It does **not** project consensus scores, construct a final analysis cohort, assess cross-dataset replication, test program concordance, or interpret methylation–expression evidence.

**Scientific boundaries.** Cohort eligibility, lineage crosswalk, donor independence, a one-record-per-case rule, full gene coverage, normalization, the replication estimand, null models and multiplicity handling must be prospectively reviewed and frozen before those analyses. Cohort source labels (`primary_site`, `disease_type`) must not be silently converted into TCGA lineages. CPTAC-3 RNA-seq eligibility must not be conditioned on DNA-methylation availability.

**Sources.** `111_cptac3_acquisition_and_audit.ipynb`, `docs/contracts/phase8/NOTEBOOK_800_PREFLIGHT_A_DRAFT.md`, `docs/NOTEBOOK_STYLE_GUIDE.md`, `config/artifact_registry.json` and `config/raw_data_registry.json`.

Run from the project's **Python 3.11.8** Jupyter environment with the local Notebook 111 outputs already generated. No GDC calls, re-download, upstream re-execution or full raw payload MD5 scan is performed.

In [ ]:
# =============================================================================
# Imports
# =============================================================================

import json
import platform

import numpy as np
import pandas as pd

from pancancer_epigenetics.utils.artifact_registry import (
    get_artifact,
    load_artifact_registry,
    resolve_artifact_path,
)
from pancancer_epigenetics.utils.file_checks import calculate_sha256
from pancancer_epigenetics.utils.paths import Paths, project_relative_path
from pancancer_epigenetics.utils.raw_data_registry import load_raw_data_registry

In [ ]:
# =============================================================================
# Notebook 111 handoff and stable Phase 4 input identifiers
# =============================================================================

AUDIT_SUMMARY_PATH = Paths.metadata / "111_cptac3_acquisition_audit_summary.json"
RAW_REGISTRY_PROPOSAL_PATH = Paths.metadata / "111_cptac3_raw_registry_proposal.json"
OUTPUT_DIR = Paths.metadata

AUDIT_INPUT_KEYS = (
    "file_index",
    "candidate_primary",
    "ambiguity_by_lineage",
)

FROZEN_ARTIFACT_IDS = {
    "gene_universe": "phase4.400.cross_system_shared_gene_universe",
    "program_catalog": "phase4.401.consensus_transcriptomic_program_catalog",
    "program_weights": "phase4.401.consensus_transcriptomic_gene_weights",
    "construction_metadata": "phase4.401.consensus_construction_metadata",
    "tcga_sample_mapping": "phase2.203.final_sample_mapping",
}

OUTPUT_PATHS = {
    "lineage_inventory": OUTPUT_DIR / "800_preflight_a_candidate_lineage_inventory.csv",
    "case_multiplicity": OUTPUT_DIR / "800_preflight_a_case_multiplicity.csv",
    "identifier_collisions": OUTPUT_DIR / "800_preflight_a_identifier_collisions.csv",
    "gene_id_pilot": OUTPUT_DIR / "800_preflight_a_gene_id_pilot.csv",
    "readiness": OUTPUT_DIR / "800_preflight_a_readiness.json",
    "rna_adjudication_case_evidence": OUTPUT_DIR / "800_preflight_b_rna_case_evidence.csv",
    "rna_adjudication_file_evidence": OUTPUT_DIR / "800_preflight_b_rna_file_evidence.csv",
}

In [ ]:
# =============================================================================
# Read Notebook 111's observed audit boundary
# =============================================================================

with AUDIT_SUMMARY_PATH.open("r", encoding="utf-8") as handle:
    audit_summary = json.load(handle)

if audit_summary.get("notebook") != 111:
    raise ValueError("The source is not a Notebook 111 audit summary")
if audit_summary.get("cohort_frozen") is not False:
    raise ValueError("Unexpected cohort-freeze flag in Notebook 111 handoff")
if audit_summary.get("phase8_validation_performed") is not False:
    raise ValueError("Unexpected Phase 8 validation status")
if audit_summary.get("n_files_by_modality") != {
    "rnaseq": 2968, "methylation": 2610,
}:
    raise ValueError("CPTAC-3 source-snapshot identity differs from Notebook 111")

print("Notebook 111 source audit: PASS (reported)")
print("Cohort freeze:", audit_summary["cohort_frozen"])
print("External validation performed:", audit_summary["phase8_validation_performed"])

In [ ]:
# =============================================================================
# Resolve and fingerprint the small derived audit tables consumed by 800
# =============================================================================

audit_inputs = {}
for name in AUDIT_INPUT_KEYS:
    record = audit_summary["audit_artifacts"][name]
    path = Paths.root / record["path"]
    if not path.is_file():
        raise FileNotFoundError(f"Missing Notebook 111 derived handoff: {path}")
    if calculate_sha256(path) != record["sha256"]:
        raise ValueError(f"Notebook 111 derived handoff changed: {name}")
    audit_inputs[name] = path

print("Verified Notebook 111 derived handoffs:")
for name, path in audit_inputs.items():
    print(" ", name, project_relative_path(path))

In [ ]:
# =============================================================================
# Read authoritative Notebook 111 tables for new downstream questions only
# =============================================================================

file_index = pd.read_csv(audit_inputs["file_index"], dtype="string")
candidate_primary = pd.read_csv(audit_inputs["candidate_primary"], dtype="string")
upstream_ambiguity = pd.read_csv(
    audit_inputs["ambiguity_by_lineage"], dtype="string"
)

new_use_columns = {
    "modality", "file_id", "filename", "case_uuid", "case_submitter_id",
    "sample_uuid", "aliquot_uuid", "mapping_status",
    "primary_site", "disease_type", "platform",
}
missing_columns = new_use_columns - set(file_index.columns)
if missing_columns:
    raise KeyError(f"Notebook 800 required columns missing: {sorted(missing_columns)}")

# Notebook 111's mapping-status distribution, source totals and primary
# sample-type inventory are authoritative and are not recalculated here.

## Source-registration boundary

The Notebook 111 registry proposal is a review artifact, **not** permission to silently modify the raw-data registry. A missing `cptac3` entry must remain visible as a registration task. Likewise, Phase 4 inputs are frozen and consumed only after exact identity checks; their weights are *not* applied to CPTAC-3 in this preflight.

In [ ]:
# =============================================================================
# Inspect CPTAC-3 raw registry state without mutating it
# =============================================================================

raw_registry = load_raw_data_registry()
with RAW_REGISTRY_PROPOSAL_PATH.open("r", encoding="utf-8") as handle:
    cptac3_registry_proposal = json.load(handle)

cptac3_registered = "cptac3" in raw_registry
print("CPTAC-3 registered:", cptac3_registered)
print("Proposed RNA status:",
      cptac3_registry_proposal["cohorts"]["rnaseq"]["status"])
print("Proposed methylation status:",
      cptac3_registry_proposal["cohorts"]["methylation"]["status"])
if cptac3_registered:
    print("Existing registration requires review against Notebook 111 proposal")

In [ ]:
# =============================================================================
# Verify exact frozen Phase 4 and TCGA reference identities
# =============================================================================

artifact_registry = load_artifact_registry()
frozen_paths = {}
for name, artifact_id in FROZEN_ARTIFACT_IDS.items():
    spec = get_artifact(artifact_registry, artifact_id)
    path = resolve_artifact_path(artifact_registry, artifact_id)
    if spec["status"] != "frozen":
        raise ValueError(f"Required upstream artifact is not frozen: {artifact_id}")
    if path.stat().st_size != spec["size_bytes"]:
        raise ValueError(f"Frozen upstream file size changed: {artifact_id}")
    if calculate_sha256(path) != spec["sha256"]:
        raise ValueError(f"Frozen upstream fingerprint changed: {artifact_id}")
    frozen_paths[name] = path

print("Verified", len(frozen_paths), "frozen input identities")

In [ ]:
# =============================================================================
# Read the frozen gene universe and consensus metadata without scoring
# =============================================================================

shared_genes = pd.read_csv(frozen_paths["gene_universe"], dtype="string")
program_catalog = pd.read_csv(frozen_paths["program_catalog"], dtype="string")
program_weights = pd.read_csv(frozen_paths["program_weights"], dtype="string")
with frozen_paths["construction_metadata"].open("r", encoding="utf-8") as handle:
    construction_metadata = json.load(handle)

required_gene_columns = {"gene_symbol", "tumor_gene_id", "cell_line_gene_id"}
if required_gene_columns - set(shared_genes.columns):
    raise KeyError("Frozen gene universe lacks identifiers required for a technical crosswalk")
if {"consensus_program_id", "gene_symbol", "consensus_weight"} - set(program_weights):
    raise KeyError("Frozen gene weights lack identity columns")

print("Frozen genes:", len(shared_genes))
print("Program IDs:", sorted(program_weights["consensus_program_id"].unique().tolist()))
print("Frozen weight rows:", len(program_weights))
print("Score standardization:", construction_metadata["score_construction"])

## Lineage-aware review of **new** case-level eligibility risks

Notebook 111 already froze technical evidence on sample ancestry, including an
`ambiguity_by_lineage` table with **file-level** denominators and a provisional
primary-tumor inventory. Notebook 800 consumes those audited numbers directly.

The new question is case-level: how many *distinct cases* in each existing
GDC stratum are touched by unresolved file ancestry, and how does that compare
descriptively with the pool of uniquely mapped provisional primary-tumor cases?
These are **not** new primary-tumor calls, new lineage mappings, or eligibility
criteria. The denominator of the unresolved-case fraction contains **all sample
types** from the acquired files, and cannot be interpreted as primary-tumor
attrition. A dedicated primary-compatible unresolved-sample sensitivity, and
the final one-case selection rule, remain separate decisions.

In [ ]:
# =============================================================================
# Reuse Notebook 111 source-stratum counts; compute only new case-level risk
# =============================================================================

STRATUM_KEYS = ["modality", "primary_site", "disease_type", "platform"]
UPSTREAM_COUNT_COLUMNS = [
    "n_files", "n_cases", "n_ambiguous_files", "ambiguous_file_fraction",
]
missing_upstream = (
    set(STRATUM_KEYS + UPSTREAM_COUNT_COLUMNS) - set(upstream_ambiguity.columns)
)
if missing_upstream:
    raise KeyError(f"111 lineage audit missing fields: {sorted(missing_upstream)}")

source_counts = upstream_ambiguity[
    STRATUM_KEYS + UPSTREAM_COUNT_COLUMNS
].copy()
for column in UPSTREAM_COUNT_COLUMNS:
    source_counts[column] = pd.to_numeric(
        source_counts[column], errors="raise"
    )

unresolved_case_counts = (
    file_index.loc[file_index["mapping_status"].eq("ambiguous_sample_ancestry")]
    .groupby(STRATUM_KEYS, dropna=False)["case_uuid"]
    .nunique()
    .rename("cases_with_ambiguous_files")
    .reset_index()
)
source_counts = source_counts.merge(
    unresolved_case_counts, on=STRATUM_KEYS, how="left", validate="one_to_one"
)
source_counts["cases_with_ambiguous_files"] = (
    source_counts["cases_with_ambiguous_files"].fillna(0).astype("int64")
)
source_counts["fraction_all_cases_with_ambiguous_files"] = (
    source_counts["cases_with_ambiguous_files"] / source_counts["n_cases"]
)

In [ ]:
# =============================================================================
# Contextualize new case-level ambiguity against the 111 primary inventory
# =============================================================================

# Notebook 111 reports primary counts by source site/disease; this new
# platform-specific case view is descriptive only and is not an eligibility gate.
primary_case_counts = (
    candidate_primary.groupby(STRATUM_KEYS, dropna=False)["case_uuid"]
    .nunique()
    .rename("provisional_primary_cases")
    .reset_index()
)

lineage_inventory = source_counts.merge(
    primary_case_counts, on=STRATUM_KEYS, how="left", validate="one_to_one"
)
lineage_inventory["provisional_primary_cases"] = (
    lineage_inventory["provisional_primary_cases"].fillna(0).astype("int64")
)
lineage_inventory = lineage_inventory.sort_values(
    ["modality", "cases_with_ambiguous_files", "primary_site", "disease_type"],
    ascending=[True, False, True, True], kind="stable"
).reset_index(drop=True)

display(lineage_inventory[
    ["modality", "primary_site", "disease_type", "platform",
     "n_cases", "cases_with_ambiguous_files",
     "fraction_all_cases_with_ambiguous_files", "provisional_primary_cases"]
].head(25))
print("Lineage crosswalk and primary-specific attrition: NOT ASSESSED")

In [ ]:
# =============================================================================
# Quantify case-level multiplicity in provisional primary records
# =============================================================================

case_multiplicity = (
    candidate_primary.groupby(["modality", "case_uuid"], dropna=False)
    .agg(
        n_files=("file_id", "nunique"),
        n_samples=("sample_uuid", "nunique"),
        n_aliquots=("aliquot_uuid", "nunique"),
        n_platforms=("platform", "nunique"),
        file_uuid_values=("file_id", lambda v: "|".join(sorted(set(v.dropna())))),
        sample_uuid_values=("sample_uuid", lambda v: "|".join(sorted(set(v.dropna())))),
        aliquot_uuid_values=("aliquot_uuid", lambda v: "|".join(sorted(set(v.dropna())))),
        primary_site_values=("primary_site", lambda v: "|".join(sorted(set(v.dropna())))),
        disease_type_values=("disease_type", lambda v: "|".join(sorted(set(v.dropna())))),
    )
    .reset_index()
)

case_multiplicity["needs_selection_policy"] = (
    case_multiplicity[["n_files", "n_samples", "n_aliquots"]].gt(1).any(axis=1)
)
case_multiplicity["selected_file_id"] = pd.Series(
    pd.NA, index=case_multiplicity.index, dtype="string"
)
case_multiplicity["selection_status"] = "NOT_FROZEN"
case_multiplicity = case_multiplicity.sort_values(
    ["modality", "case_uuid"], kind="stable"
).reset_index(drop=True)

display(
    case_multiplicity.groupby("modality")
    .agg(
        candidate_cases=("case_uuid", "nunique"),
        cases_needing_review=("needs_selection_policy", "sum"),
        maximum_files_per_case=("n_files", "max"),
        maximum_samples_per_case=("n_samples", "max"),
    )
)
print("No case-level selection, averaging or technical-replicate pooling performed")

## RNA-seq per-case multiplicity: outcome-blind structural classification

The initial 800 preflight identified **288 provisional RNA-seq cases requiring multiplicity review**. The next technical task is to distinguish case-level identifier structures **without** reopening Notebook 111's file-to-sample ancestry audit or making a biological sample-selection decision.

Classification is based only on already audited file, sample, and aliquot UUIDs. **More than one file on a single aliquot is not, by itself, proof of technical replication or equivalence of processed files.** More than one aliquot under one sample is not automatically exchangeable, and distinct sample UUIDs under one case cannot be pooled as independent patients.

Cases with internally inconsistent counts or cross-file mapping conflicts must remain explicitly unresolved. Even a case with one file is merely *technically simple*, **not** finally eligible: donor independence, lineage harmonization, technical QC and prospective Phase 8 scoring policy remain open.

In [ ]:
# =============================================================================
# Classify RNA-seq case multiplicity from the existing 800 case inventory
# =============================================================================

rna_case_mask = case_multiplicity["modality"].eq("rnaseq")
rna_cases = case_multiplicity.loc[rna_case_mask].copy()

inconsistent_counts = (
    rna_cases[["n_files", "n_samples", "n_aliquots"]].le(0).any(axis=1)
    | rna_cases["n_aliquots"].gt(rna_cases["n_files"])
    | rna_cases["n_samples"].gt(rna_cases["n_aliquots"])
)
rna_cases["rna_multiplicity_class"] = np.select(
    [
        inconsistent_counts,
        rna_cases["n_samples"].gt(1),
        rna_cases["n_aliquots"].gt(1),
        rna_cases["n_files"].gt(1),
    ],
    [
        "INCONSISTENT_BIOSPECIMEN_COUNTS",
        "MULTIPLE_SAMPLES_IN_CASE",
        "ONE_SAMPLE_MULTIPLE_ALIQUOTS",
        "ONE_ALIQUOT_MULTIPLE_FILES",
    ],
    default="SINGLE_FILE_SAMPLE_ALIQUOT",
)
print("RNA-seq provisional cases classified:", len(rna_cases))
print("No record selected, pooled, excluded or scored")

In [ ]:
# =============================================================================
# Screen for cross-file ancestry conflicts not resolved by single-file links
# =============================================================================

rna_candidate_rows = candidate_primary.loc[
    candidate_primary["modality"].eq("rnaseq"),
    ["case_uuid", "sample_uuid", "aliquot_uuid", "file_id"],
].copy()

aliquot_links = (
    rna_candidate_rows.groupby("aliquot_uuid")
    .agg(
        n_parent_samples=("sample_uuid", "nunique"),
        n_parent_cases=("case_uuid", "nunique"),
    )
)
sample_links = (
    rna_candidate_rows.groupby("sample_uuid")
    .agg(n_parent_cases=("case_uuid", "nunique"))
)
conflicting_aliquots = set(aliquot_links.loc[
    aliquot_links["n_parent_samples"].gt(1)
    | aliquot_links["n_parent_cases"].gt(1)
].index)
conflicting_samples = set(sample_links.loc[
    sample_links["n_parent_cases"].gt(1)
].index)

rna_cases["cross_file_ancestry_conflict"] = (
    rna_cases["aliquot_uuid_values"].map(
        lambda s: any(part in conflicting_aliquots for part in s.split("|"))
    )
    | rna_cases["sample_uuid_values"].map(
        lambda s: any(part in conflicting_samples for part in s.split("|"))
    )
)
rna_cases.loc[
    rna_cases["cross_file_ancestry_conflict"],
    "rna_multiplicity_class",
] = "CROSS_FILE_BIOSPECIMEN_CONFLICT"

print("Aliquot UUIDs with incompatible cross-file parents:", len(conflicting_aliquots))
print("Sample UUIDs with incompatible cross-file cases:", len(conflicting_samples))
print("Affected RNA cases:", int(rna_cases["cross_file_ancestry_conflict"].sum()))

In [ ]:
# =============================================================================
# Show nonoverlapping RNA-seq structural classes and review candidates
# =============================================================================

rna_class_summary = (
    rna_cases.groupby("rna_multiplicity_class", dropna=False)
    .agg(
        n_cases=("case_uuid", "nunique"),
        max_files_in_case=("n_files", "max"),
        max_samples_in_case=("n_samples", "max"),
    )
    .reset_index()
    .sort_values(["n_cases", "rna_multiplicity_class"], ascending=[False, True])
)
rna_class_summary["fraction_provisional_rna_cases"] = (
    rna_class_summary["n_cases"] / len(rna_cases)
)

display(rna_class_summary)
print(
    "Cases previously flagged for multiplicity review:",
    int(rna_cases["needs_selection_policy"].sum()),
)
print("Unresolved case selection: ALL — no file selection rule frozen")

# Preserve exact source labels; mixed labels must remain visible.
display(
    rna_cases.loc[rna_cases["needs_selection_policy"]]
    .groupby(["primary_site_values", "rna_multiplicity_class"], dropna=False)
    .size().rename("n_cases").reset_index()
    .sort_values(["n_cases", "primary_site_values"], ascending=[False, True])
    .head(20)
)

In [ ]:
# =============================================================================
# Annotate the existing case-level output; create no duplicate source inventory
# =============================================================================

case_multiplicity["rna_multiplicity_class"] = pd.Series(
    pd.NA, index=case_multiplicity.index, dtype="string"
)
case_multiplicity["rna_cross_file_ancestry_conflict"] = pd.Series(
    pd.NA, index=case_multiplicity.index, dtype="boolean"
)

case_multiplicity.loc[
    rna_case_mask, "rna_multiplicity_class"
] = rna_cases["rna_multiplicity_class"].to_numpy()
case_multiplicity.loc[
    rna_case_mask, "rna_cross_file_ancestry_conflict"
] = rna_cases["cross_file_ancestry_conflict"].to_numpy()

if (
    rna_class_summary["n_cases"].sum() != len(rna_cases)
    or case_multiplicity.loc[rna_case_mask, "rna_multiplicity_class"].isna().any()
):
    raise ValueError("Incomplete RNA-seq case multiplicity classification")

print("Annotated output:", project_relative_path(OUTPUT_PATHS["case_multiplicity"]))
print("Classification: descriptive only; no sample selected")

## PREFLIGHT-B — Outcome-blind RNA-seq multiplicity adjudication evidence

**Question:** for the 288 cases containing multiple uniquely mapped provisional primary-tumor RNA-seq files/samples/aliquots, which *existing, audited* source metadata can distinguish candidate biospecimens? Does that information supply a defensible basis for selecting one observation per case?

This is a **metadata evidence audit**, not an actual adjudication. It consumes the already classified RNA cases and Notebook 111's frozen candidate file index; it does not rerun source acquisition, reopen ancestral assignment, reinterpret ambiguous files, or inspect any gene expression or program scores.

The preserved GDC descriptors `sample_type`, `tissue_type` and `tumor_descriptor` may identify disagreements but cannot resolve tumor-region/timepoint differences by themselves. Sample/aliquot submitter IDs are identifiers, not evidence of relative quality. Provider file size, MD5 and generic STAR workflow are **not sequencing QC selection metrics**. Even identical submitted labels do not prove samples are interchangeable.

All 288 case/file identities remain in a **NOT_FROZEN** state. Any future one-per-case decision must use a separately reviewed outcome-blind policy, with enough objective specimen provenance and technical QC. Cases without a defensible selection basis remain unresolved; never select on UUID ordering, file size, gene expression or program score.

In [ ]:
# =============================================================================
# Restrict new evidence review to the 288 RNA cases already identified in 800
# =============================================================================

rna_review_cases = rna_cases.loc[
    rna_cases["needs_selection_policy"], [
        "case_uuid", "rna_multiplicity_class", "n_files", "n_samples",
        "n_aliquots", "cross_file_ancestry_conflict",
    ]
].copy()

RNA_SOURCE_EVIDENCE_COLS = [
    "case_uuid", "file_id", "filename", "sample_uuid", "sample_submitter_id",
    "aliquot_uuid", "aliquot_submitter_id", "sample_type", "tissue_type",
    "tumor_descriptor", "primary_site", "disease_type", "project_id",
    "workflow_type", "platform", "size_bytes", "provider_md5",
]
missing = set(RNA_SOURCE_EVIDENCE_COLS) - set(candidate_primary.columns)
if missing:
    raise KeyError("Required existing Notebook 111 metadata absent: " + str(sorted(missing)))

rna_adjudication_files = candidate_primary.loc[
    candidate_primary["modality"].eq("rnaseq")
    & candidate_primary["case_uuid"].isin(rna_review_cases["case_uuid"]),
    RNA_SOURCE_EVIDENCE_COLS,
].copy()

if (
    rna_review_cases["case_uuid"].duplicated().any()
    or rna_adjudication_files["file_id"].duplicated().any()
    or set(rna_adjudication_files["case_uuid"]) != set(rna_review_cases["case_uuid"])
    or len(rna_adjudication_files) != int(rna_review_cases["n_files"].sum())
):
    raise ValueError("800-B candidate review does not reconcile to 800-A case inventory")

print("Cases under metadata adjudication review:", len(rna_review_cases))
print("Existing primary RNA file records under review:", len(rna_adjudication_files))

In [ ]:
# =============================================================================
# Quantify per-case specimen descriptors and source-file metadata coverage
# =============================================================================

def nonnull_count(series):
    return int(series.notna().sum())

def unique_nonnull_count(series):
    return int(series.nunique(dropna=True))

evidence = (
    rna_adjudication_files.groupby("case_uuid", dropna=False)
    .agg(
        observed_files=("file_id", "nunique"),
        observed_samples=("sample_uuid", "nunique"),
        observed_aliquots=("aliquot_uuid", "nunique"),
        sample_submitter_ids=("sample_submitter_id", unique_nonnull_count),
        sample_submitter_id_file_coverage=("sample_submitter_id", nonnull_count),
        aliquot_submitter_ids=("aliquot_submitter_id", unique_nonnull_count),
        aliquot_submitter_id_file_coverage=("aliquot_submitter_id", nonnull_count),
        tissue_type_values=("tissue_type", unique_nonnull_count),
        tissue_type_file_coverage=("tissue_type", nonnull_count),
        tumor_descriptor_values=("tumor_descriptor", unique_nonnull_count),
        tumor_descriptor_file_coverage=("tumor_descriptor", nonnull_count),
        source_site_values=("primary_site", unique_nonnull_count),
        source_disease_values=("disease_type", unique_nonnull_count),
        project_values=("project_id", unique_nonnull_count),
        workflow_values=("workflow_type", unique_nonnull_count),
        platform_values=("platform", unique_nonnull_count),
        distinct_provider_md5=("provider_md5", unique_nonnull_count),
    ).reset_index()
)
rna_adjudication_cases = rna_review_cases.merge(
    evidence, on="case_uuid", how="left", validate="one_to_one"
)

if not (
    rna_adjudication_cases["n_files"].eq(rna_adjudication_cases["observed_files"]).all()
    and rna_adjudication_cases["n_samples"].eq(
        rna_adjudication_cases["observed_samples"]
    ).all()
    and rna_adjudication_cases["n_aliquots"].eq(
        rna_adjudication_cases["observed_aliquots"]
    ).all()
):
    raise ValueError("New specimen evidence does not match prior case structure")

display(
    rna_adjudication_cases[[
        "rna_multiplicity_class", "observed_files",
        "tissue_type_file_coverage", "tumor_descriptor_file_coverage",
        "sample_submitter_id_file_coverage",
        "aliquot_submitter_id_file_coverage",
    ]].groupby("rna_multiplicity_class").sum()
)

In [ ]:
# =============================================================================
# Flag metadata discordance and insufficiency, without resolving samples
# =============================================================================

rna_adjudication_cases["source_descriptor_discordance"] = (
    rna_adjudication_cases[[
        "tissue_type_values", "tumor_descriptor_values",
        "source_site_values", "source_disease_values",
        "project_values", "workflow_values", "platform_values",
    ]].gt(1).any(axis=1)
)
rna_adjudication_cases["source_descriptor_incomplete"] = (
    rna_adjudication_cases["tissue_type_file_coverage"].lt(
        rna_adjudication_cases["n_files"]
    )
    | rna_adjudication_cases["tumor_descriptor_file_coverage"].lt(
        rna_adjudication_cases["n_files"]
    )
    | rna_adjudication_cases["sample_submitter_id_file_coverage"].lt(
        rna_adjudication_cases["n_files"]
    )
    | rna_adjudication_cases["aliquot_submitter_id_file_coverage"].lt(
        rna_adjudication_cases["n_files"]
    )
)
rna_adjudication_cases["same_md5_across_distinct_files"] = (
    rna_adjudication_cases["distinct_provider_md5"].lt(
        rna_adjudication_cases["n_files"]
    )
)

rna_adjudication_cases["adjudication_evidence_class"] = np.select(
    [
        rna_adjudication_cases["cross_file_ancestry_conflict"],
        rna_adjudication_cases["source_descriptor_discordance"],
        rna_adjudication_cases["source_descriptor_incomplete"],
        rna_adjudication_cases["rna_multiplicity_class"].eq(
            "MULTIPLE_SAMPLES_IN_CASE"
        ),
    ],
    [
        "CROSS_FILE_CONFLICT_UNRESOLVED",
        "SOURCE_DESCRIPTORS_DISCORDANT_REVIEW",
        "SOURCE_DESCRIPTORS_INCOMPLETE_REVIEW",
        "MULTIPLE_SAMPLES_NO_QC_SELECTION_BASIS",
    ],
    default="MULTIPLE_ALIQUOTS_OR_FILES_NO_QC_SELECTION_BASIS",
)
rna_adjudication_cases["selection_evidence_status"] = (
    "INSUFFICIENT_TO_SELECT_WITHOUT_PROSPECTIVE_QC_POLICY"
)
rna_adjudication_cases["selected_file_id"] = pd.Series(
    pd.NA, index=rna_adjudication_cases.index, dtype="string"
)

display(
    rna_adjudication_cases["adjudication_evidence_class"]
    .value_counts(dropna=False)
    .rename_axis("evidence_class").reset_index(name="n_cases")
)
print("No files selected or excluded; no technical QC read from STAR payloads")

In [ ]:
# =============================================================================
# Describe the evidence limits by source tumor context without rerunning 111
# =============================================================================

rna_case_context = rna_cases.loc[
    rna_cases["needs_selection_policy"],
    ["case_uuid", "primary_site_values", "disease_type_values"],
].copy()
rna_adjudication_cases = rna_adjudication_cases.merge(
    rna_case_context, on="case_uuid", validate="one_to_one"
)
display(
    rna_adjudication_cases.groupby(
        ["primary_site_values", "rna_multiplicity_class"], dropna=False
    ).agg(
        n_cases=("case_uuid", "nunique"),
        descriptor_discordance=("source_descriptor_discordance", "sum"),
        incomplete_descriptors=("source_descriptor_incomplete", "sum"),
        same_md5_file_groups=("same_md5_across_distinct_files", "sum"),
    )
    .reset_index()
    .sort_values(["n_cases", "primary_site_values"], ascending=[False, True])
    .head(25)
)
print(
    "Cases with source-descriptor disagreement:",
    int(rna_adjudication_cases["source_descriptor_discordance"].sum())
)
print(
    "Cases with at least one missing specimen descriptor field:",
    int(rna_adjudication_cases["source_descriptor_incomplete"].sum())
)
print("True technical replicate identity / relative RNA-seq QC: NOT ASSESSED")

In [ ]:
# =============================================================================
# Prepare auditable per-file metadata evidence; preserve case status UNRESOLVED
# =============================================================================

rna_adjudication_files = rna_adjudication_files.merge(
    rna_adjudication_cases[
        ["case_uuid", "rna_multiplicity_class", "adjudication_evidence_class",
         "selection_evidence_status"]
    ],
    on="case_uuid", validate="many_to_one"
)
rna_adjudication_files["candidate_selected"] = False
rna_adjudication_files["selection_performed"] = False

rna_adjudication_cases = rna_adjudication_cases.sort_values(
    "case_uuid", kind="stable"
).reset_index(drop=True)
rna_adjudication_files = rna_adjudication_files.sort_values(
    ["case_uuid", "sample_uuid", "aliquot_uuid", "file_id"], kind="stable"
).reset_index(drop=True)

if (
    len(rna_adjudication_cases) != int(rna_cases["needs_selection_policy"].sum())
    or rna_adjudication_cases["selected_file_id"].notna().any()
    or rna_adjudication_files["candidate_selected"].any()
):
    raise ValueError("Evidence-only 800-B unexpectedly selected or lost cases")

print("Case-level adjudication evidence rows:", len(rna_adjudication_cases))
print("File-level adjudication evidence rows:", len(rna_adjudication_files))
print("Selection eligibility rule: NOT_FROZEN")

## Lexical identifier-overlap audit

Compare exact GDC case UUID and submitter strings against the frozen TCGA cohort. This is a **collision screen only**. Separate projects may assign different IDs to the same biological donor; therefore an absence of collisions is not evidence of documented donor independence. A provenance/crosswalk review remains a gate before confirmatory replication claims.

In [ ]:
# =============================================================================
# Read TCGA source identities from the frozen Phase 2 sample mapping
# =============================================================================

rna_primary = candidate_primary.loc[
    candidate_primary["modality"].eq("rnaseq")
].copy()

tcga_mapping = pd.read_csv(frozen_paths["tcga_sample_mapping"], dtype="string")
tcga_required = {"rna_case_uuid", "case_submitter_id"}
if tcga_required - set(tcga_mapping.columns):
    raise KeyError("Frozen TCGA mapping lacks required case identifiers")

tcga_case_uuid_set = set(tcga_mapping["rna_case_uuid"].dropna())
tcga_submitter_set = set(tcga_mapping["case_submitter_id"].dropna())

print("Candidate CPTAC-3 RNA cases:", rna_primary["case_uuid"].nunique())
print("Donor independence: UNRESOLVED without source crosswalk")

In [ ]:
# =============================================================================
# Create a case-level exact-string collision screen, not independence labels
# =============================================================================

identifier_collisions = (
    candidate_primary.groupby(["modality", "case_uuid"], dropna=False)
    .agg(
        n_submitter_ids=("case_submitter_id", "nunique"),
        submitter_id_values=(
            "case_submitter_id",
            lambda v: "|".join(sorted(set(v.dropna()))),
        ),
        primary_site_values=(
            "primary_site", lambda v: "|".join(sorted(set(v.dropna())))
        ),
        disease_type_values=(
            "disease_type", lambda v: "|".join(sorted(set(v.dropna())))
        ),
    )
    .reset_index()
)
identifier_collisions["tcga_case_uuid_collision"] = (
    identifier_collisions["case_uuid"].isin(tcga_case_uuid_set)
)
identifier_collisions["tcga_submitter_collision"] = (
    identifier_collisions["submitter_id_values"].map(
        lambda s: any(v in tcga_submitter_set for v in s.split("|") if v)
    )
)
identifier_collisions["donor_independence_status"] = (
    "UNRESOLVED_WITHOUT_EXTERNAL_PROVENANCE"
)
identifier_collisions["case_submitter_ambiguity"] = (
    identifier_collisions["n_submitter_ids"].gt(1)
)

display(
    identifier_collisions.groupby("modality")
    .agg(
        candidate_cases=("case_uuid", "nunique"),
        exact_uuid_collision_rows=("tcga_case_uuid_collision", "sum"),
        exact_submitter_collision_rows=("tcga_submitter_collision", "sum"),
    )
)

## Pilot technical gene-ID compatibility

The source library has already passed Notebook 111 file-integrity checks. A single deterministically selected *provisional primary RNA-seq* file is opened here **only** to inspect the GDC STAR Counts header and Ensembl identifier mapping to the 2,389-gene frozen universe.

This is a technical pilot, **not** a complete feature-coverage assessment, cohort normalization, or program test. A match after stripping Ensembl versions is reported only as a diagnostic; no version stripping, symbol fallback, missing-gene imputation or rescue is applied. Full per-file/lineage compatibility and thresholds remain open until reviewed.

In [ ]:
# =============================================================================
# Select one deterministic RNA-seq technical pilot from the frozen source files
# =============================================================================

if rna_primary.empty:
    raise ValueError("Notebook 111 has no provisional primary RNA-seq candidates")
pilot_record = rna_primary.sort_values(
    ["file_id", "filename"], kind="stable"
).iloc[0]
pilot_file_uuid = pilot_record["file_id"]
pilot_path = (
    Paths.root / "data" / "raw" / "cptac3" / "rnaseq_star_counts"
    / pilot_file_uuid / pilot_record["filename"]
)
if not pilot_path.is_file():
    raise FileNotFoundError(f"Frozen Notebook 111 pilot payload is missing: {pilot_path}")

pilot_header = pd.read_csv(pilot_path, sep="\t", comment="#", nrows=0).columns.tolist()
print("Pilot GDC file:", pilot_file_uuid)
print("STAR Counts fields:", pilot_header)
if not {"gene_id", "unstranded"}.issubset(pilot_header):
    raise ValueError("Unexpected STAR Counts schema; no gene crosswalk attempted")

In [ ]:
# =============================================================================
# Audit exact/version-insensitive Ensembl ID coverage in the technical pilot
# =============================================================================

pilot_gene_ids = pd.read_csv(
    pilot_path, sep="\t", comment="#", usecols=["gene_id"], dtype="string"
)["gene_id"].dropna()
provider_ids = set(pilot_gene_ids.tolist())
provider_versions_stripped = pilot_gene_ids.str.replace(
    r"\.\d+$", "", regex=True
)
base_id_counts = provider_versions_stripped.value_counts()
ambiguous_base_ids = set(base_id_counts[base_id_counts.gt(1)].index)

gene_id_pilot = shared_genes[
    ["gene_symbol", "tumor_gene_id", "cell_line_gene_id"]
].copy()
gene_id_pilot["pilot_file_id"] = pilot_file_uuid
gene_id_pilot["exact_ensembl_match"] = (
    gene_id_pilot["tumor_gene_id"].isin(provider_ids)
)
gene_id_pilot["versionless_pilot_match_diagnostic"] = (
    gene_id_pilot["tumor_gene_id"].str.replace(r"\.\d+$", "", regex=True)
    .isin(set(provider_versions_stripped))
)
gene_id_pilot["versionless_pilot_id_ambiguous"] = (
    gene_id_pilot["tumor_gene_id"].str.replace(r"\.\d+$", "", regex=True)
    .isin(ambiguous_base_ids)
)
gene_id_pilot["mapping_applied"] = False

display(gene_id_pilot[
    ["exact_ensembl_match", "versionless_pilot_match_diagnostic",
     "versionless_pilot_id_ambiguous"]
].sum().to_frame(name="genes"))
print("Cohort-wide feature compatibility: NOT ASSESSED")

## Readiness and persistence

The preflight outputs remain local, provisional diagnostics under `data/interim/metadata`. They must not be registered as frozen biological validation evidence or used for Phase 9. The RNA-seq per-case selection and lineage crosswalk remain unresolved; project-specific donor independence remains unverified; the statistical Phase 8 analysis contract is not frozen.

New outputs receive basic shape/column validation after persistence, while Notebook 111's prior raw MD5 scan, upstream source inventories and frozen program-construction logic are not rerun.

In [ ]:
# =============================================================================
# Assemble explicit readiness statuses and input provenance
# =============================================================================

input_provenance = {
    "111_summary": {
        "path": project_relative_path(AUDIT_SUMMARY_PATH),
        "sha256": calculate_sha256(AUDIT_SUMMARY_PATH),
    },
    "111_registry_proposal": {
        "path": project_relative_path(RAW_REGISTRY_PROPOSAL_PATH),
        "sha256": calculate_sha256(RAW_REGISTRY_PROPOSAL_PATH),
    },
}
input_provenance.update({
    "111_" + name: {
        "path": project_relative_path(path),
        "sha256": audit_summary["audit_artifacts"][name]["sha256"],
    }
    for name, path in audit_inputs.items()
})
input_provenance.update({
    artifact_id: {
        "path": get_artifact(artifact_registry, artifact_id)["path"],
        "sha256": get_artifact(artifact_registry, artifact_id)["sha256"],
    }
    for artifact_id in FROZEN_ARTIFACT_IDS.values()
})

readiness_status = {
    "notebook111_source_integrity": "UPSTREAM_PASS_REUSED",
    "notebook111_local_handoff_identity": "PASS",
    "phase4_frozen_identity": "PASS",
    "provisional_primary_inventory": "AVAILABLE_NOT_COHORT_FROZEN",
    "one_record_per_case": "PENDING_POLICY",
    "rna_multiplicity_adjudication_evidence": "DESCRIBED_UNRESOLVED_NO_SELECTION",
    "rna_case_multiplicity_classes": "DESCRIPTIVE_COMPLETE_NOT_RESOLVED",
    "rna_cross_file_ancestry_conflicts": (
        "REVIEW_REQUIRED" if rna_cases["cross_file_ancestry_conflict"].any()
        else "NONE_DETECTED_WITHIN_PROVISIONAL_PRIMARY_FILES"
    ),
    "harmonized_lineage_crosswalk": "NOT_ASSESSED",
    "donor_independence": "UNRESOLVED",
    "star_counts_gene_compatibility": "PILOT_ONLY",
    "full_transcriptomic_qc_and_coverage": "NOT_ASSESSED",
    "methylation_platform_and_sample_harmonization": "NOT_ASSESSED",
    "external_replication_analysis_contract": "NOT_FROZEN",
    "biological_validation_authorized": False,
}
display(pd.DataFrame(
    [{"gate": gate, "status": status} for gate, status in readiness_status.items()]
))

In [ ]:
# =============================================================================
# Persist new technical inventories; leave registries and raw data untouched
# =============================================================================

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
output_tables = {
    "lineage_inventory": lineage_inventory,
    "case_multiplicity": case_multiplicity,
    "identifier_collisions": identifier_collisions,
    "gene_id_pilot": gene_id_pilot,
    "rna_adjudication_case_evidence": rna_adjudication_cases,
    "rna_adjudication_file_evidence": rna_adjudication_files,
}
for name, frame in output_tables.items():
    frame.to_csv(OUTPUT_PATHS[name], index=False)
    print(name, "→", project_relative_path(OUTPUT_PATHS[name]), "rows:", len(frame))

In [ ]:
# =============================================================================
# Validate newly persisted technical inventories, not all upstream schemas
# =============================================================================

for name, frame in output_tables.items():
    reloaded = pd.read_csv(OUTPUT_PATHS[name], dtype="string")
    if len(reloaded) != len(frame) or list(reloaded.columns) != list(frame.columns):
        raise ValueError(f"New preflight output is inconsistent: {name}")

print("New technical inventory persistence: PASS")

In [ ]:
# =============================================================================
# Write the non-frozen readiness record with provenance and output fingerprints
# =============================================================================

readiness_record = {
    "notebook": "800_external_bulk_cohorts",
    "stage": "PREFLIGHT_A_AND_B_METADATA_EVIDENCE_ONLY",
    "environment": {
        "python": platform.python_version(),
        "pandas": pd.__version__,
        "numpy": np.__version__,
    },
    "upstream_111_snapshot": audit_summary["source_fingerprints"],
    "inputs": input_provenance,
    "raw_registry_cptac3_present": cptac3_registered,
    "source_count_context": audit_summary["n_files_by_modality"],
    "program_gene_universe_n": len(shared_genes),
    "rna_case_multiplicity_class_counts": {
        str(row.rna_multiplicity_class): int(row.n_cases)
        for row in rna_class_summary.itertuples(index=False)
    },
    "rna_preflight_b_evidence_class_counts": {
        str(k): int(v) for k, v in
        rna_adjudication_cases["adjudication_evidence_class"]
        .value_counts(dropna=False).items()
    },
    "rna_preflight_b_case_count": len(rna_adjudication_cases),
    "rna_preflight_b_descriptor_discordance_cases": int(
        rna_adjudication_cases["source_descriptor_discordance"].sum()
    ),
    "rna_preflight_b_descriptor_incomplete_cases": int(
        rna_adjudication_cases["source_descriptor_incomplete"].sum()
    ),
    "rna_preflight_b_selection_performed": False,
    "pilot_file_uuid": str(pilot_file_uuid),
    "pilot_star_counts_fields": pilot_header,
    "readiness": readiness_status,
    "cohort_frozen": False,
    "program_scores_computed": False,
    "external_replication_test_performed": False,
    "outputs": {
        name: {
            "path": project_relative_path(OUTPUT_PATHS[name]),
            "rows": len(output_tables[name]),
            "sha256": calculate_sha256(OUTPUT_PATHS[name]),
        }
        for name in output_tables
    },
}
with OUTPUT_PATHS["readiness"].open("w", encoding="utf-8") as handle:
    json.dump(readiness_record, handle, indent=2, ensure_ascii=False)
    handle.write("\n")
with OUTPUT_PATHS["readiness"].open("r", encoding="utf-8") as handle:
    persisted_readiness = json.load(handle)
if persisted_readiness["readiness"] != readiness_status:
    raise ValueError("Persisted readiness differs from the in-memory record")
print("Readiness saved:", project_relative_path(OUTPUT_PATHS["readiness"]))

## PREFLIGHT-A checkpoint — review required before Phase 8 replication

**Technical evidence only.** This notebook records whether Notebook 111's local derived handoffs and frozen Phase 4 inputs are readable and unchanged, and prepares detailed diagnostics of the provisional CPTAC-3 candidate cohort.

**Still open:** external donor independence beyond lexical ID collisions; lineage-specific attribution of sample-ancestry attrition; primary case-level selection; per-file STAR gene coverage and metadata/workflow QC; normalization/reference scaling; independent program-replication estimand and corresponding null; sample- and platform-aware methylation context; prospective multiplicity and inference policy.

**Next decision:** review the new technical tables and the Phase 8 preflight draft before freezing any cohort or analysis contract. A successful Run All here does *not* authorize biological replication testing or prove cohort readiness. Positive, negative and not-evaluable results must later be reported under the same frozen inferential design.

**PREFLIGHT-B checkpoint — no file adjudication yet.** The 288 RNA-seq multiplicity cases are reviewed only using already audited source-file and biospecimen descriptors. New per-case and per-file evidence inventories preserve source UUIDs, missing/discordant metadata and the absence of a defensible technical QC selection rule. Neither a case flagged for discordance nor a technically simple case is automatically excluded or included. Final one-record-per-case selection, replicate equivalence, raw RNA-seq QC and specimen provenance remain unassessed. No output belongs to frozen validation results.

**Additional case-level multiplicity audit (outcome-blind).** The RNA-seq provisional case inventory now reports mutually exclusive identifier-structure classes, including cross-file ancestry conflicts, without requiring re-execution of Notebook 111, promoting any file, or changing `NOT_FROZEN` selection status. The case-multiplicity CSV gains class/contradiction fields; the readiness JSON records class counts. Actual classes must be reviewed after local Run All; no category is a final analytic-eligibility decision.